# GPU Memory Profiling & Optimization — Every Byte Accounted For

**Module 26** of the PyTorch Complete Learning Guide

This notebook walks through:
- Where GPU memory goes during training
- Estimating memory requirements without a GPU (meta device)
- Using PyTorch's memory monitoring tools
- Memory snapshots and leak detection
- Every optimization technique in the toolkit

> Most cells run on **CPU only**. GPU-specific tools are clearly marked.

In [ ]:
import math
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch {torch.__version__}")
HAS_CUDA = torch.cuda.is_available()
print(f"CUDA available: {HAS_CUDA}")
if HAS_CUDA:
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Where Does GPU Memory Go?

During training, GPU memory is consumed by five main categories:

| Category | Formula | 7B bf16 Example |
|----------|---------|------------------|
| **Parameters** | `num_params × bytes_per_param` | 7B × 2 = **14 GB** |
| **Gradients** | `num_params × bytes_per_param` | 7B × 2 = **14 GB** |
| **Optimizer (Adam)** | `num_params × 4 × 2` (m + v in fp32) | 7B × 8 = **56 GB** |
| **Master weights** | `num_params × 4` (fp32 copy) | 7B × 4 = **28 GB** |
| **Activations** | `∝ batch × seq × hidden × layers` | **~12-20 GB** |
| **CUDA context** | Fixed overhead | **~0.3-0.5 GB** |
| | **Total** | **~125-133 GB** |

The optimizer state often dominates! Adam stores **2× parameter count** in fp32 state.

## 2. Meta Device Analysis — Estimate Memory Without a GPU

The `meta` device creates tensors with shapes and dtypes but **no actual storage**.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn_up = nn.Linear(dim, 4 * dim, bias=False)
        self.ffn_down = nn.Linear(4 * dim, dim, bias=False)

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + h
        x = x + self.ffn_down(F.gelu(self.ffn_up(self.norm2(x))))
        return x


class SimpleLLM(nn.Module):
    def __init__(self, vocab=32000, dim=4096, heads=32, layers=32):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.layers = nn.ModuleList([TransformerBlock(dim, heads) for _ in range(layers)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))

In [ ]:
# Build on meta device — zero actual memory used
with torch.device('meta'):
    model = SimpleLLM(vocab=32000, dim=4096, heads=32, layers=32)

total_params = sum(p.numel() for p in model.parameters())
fp32_gb = sum(p.numel() * 4 for p in model.parameters()) / 1e9
bf16_gb = sum(p.numel() * 2 for p in model.parameters()) / 1e9

print(f"Parameters:  {total_params:>15,}")
print(f"fp32 size:   {fp32_gb:>12.2f} GB")
print(f"bf16 size:   {bf16_gb:>12.2f} GB")
print()

# Per-component breakdown
for name, child in model.named_children():
    n = sum(p.numel() for p in child.parameters())
    pct = 100.0 * n / total_params
    print(f"  {name:10s}: {n:>12,} params ({pct:5.1f}%)")

del model

## 3. "Will It Fit?" Calculator

Estimate total training memory and check against common GPU configs.

In [ ]:
def estimate_training_memory(
    num_params, dtype_bytes=2, optimizer="adam",
    batch_size=1, seq_len=2048, hidden_dim=4096,
    num_layers=32, num_heads=32,
    flash_attn=True, grad_ckpt=False,
):
    param_gb = num_params * dtype_bytes / 1e9
    grad_gb = param_gb

    if optimizer == "sgd":
        opt_gb = num_params * 4 / 1e9
    elif optimizer == "adam":
        opt_gb = num_params * 4 * 2 / 1e9
        if dtype_bytes < 4:
            opt_gb += num_params * 4 / 1e9  # master weights
    elif optimizer == "adam_8bit":
        opt_gb = num_params * 2 / 1e9
        if dtype_bytes < 4:
            opt_gb += num_params * 4 / 1e9
    else:
        opt_gb = 0

    per_layer = 2 * batch_size * seq_len * hidden_dim * dtype_bytes
    if flash_attn:
        attn = batch_size * num_heads * seq_len * 64 * dtype_bytes
    else:
        attn = batch_size * num_heads * seq_len * seq_len * dtype_bytes

    eff_layers = int(num_layers ** 0.5) if grad_ckpt else num_layers
    act_gb = eff_layers * (per_layer + attn) / 1e9

    total = param_gb + grad_gb + opt_gb + act_gb + 0.5  # 0.5 GB CUDA context

    return {
        "params_gb": param_gb, "grads_gb": grad_gb,
        "optimizer_gb": opt_gb, "activations_gb": act_gb,
        "cuda_ctx_gb": 0.5, "total_gb": total,
    }


GPU_MEM = {
    "RTX-4090": 24, "A100-40GB": 40, "A100-80GB": 80,
    "H100-80GB": 80, "H200-141GB": 141,
}


def will_it_fit(est, gpu="A100-80GB"):
    avail = GPU_MEM.get(gpu, 80) * 0.9
    return est["total_gb"] <= avail

In [ ]:
# Try different model sizes
scenarios = [
    ("1B bf16 Adam", dict(num_params=1e9, optimizer="adam", batch_size=8)),
    ("7B bf16 Adam", dict(num_params=7e9, optimizer="adam", batch_size=4)),
    ("7B bf16 Adam + ckpt", dict(num_params=7e9, optimizer="adam", batch_size=4, grad_ckpt=True)),
    ("13B bf16 Adam", dict(num_params=13e9, optimizer="adam", batch_size=4,
                           hidden_dim=5120, num_layers=40, num_heads=40)),
]

for label, kwargs in scenarios:
    est = estimate_training_memory(**{k: int(v) if isinstance(v, float) and v > 100 else v
                                     for k, v in kwargs.items()})
    print(f"\n{label}:")
    print(f"  Params: {est['params_gb']:.1f} GB | Grads: {est['grads_gb']:.1f} GB | "
          f"Optimizer: {est['optimizer_gb']:.1f} GB | Activations: {est['activations_gb']:.1f} GB")
    print(f"  TOTAL: {est['total_gb']:.1f} GB")
    for gpu in ["RTX-4090", "A100-80GB", "H100-80GB", "H200-141GB"]:
        fits = will_it_fit(est, gpu)
        print(f"    {gpu}: {'YES' if fits else 'NO — need FSDP'}")

## 4. Memory Monitoring Tools

PyTorch provides several functions to monitor GPU memory at runtime.

| Function | Returns |
|----------|--------|
| `memory_allocated()` | Bytes actively used by tensors |
| `memory_reserved()` | Bytes held by caching allocator |
| `max_memory_allocated()` | Peak allocated (high-water mark) |
| `memory_summary()` | Detailed table |
| `memory_stats()` | Dict of all stats (for logging) |

In [ ]:
if HAS_CUDA:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    def mb(b): return b / 1024**2

    print("=== memory_allocated vs memory_reserved ===")
    x = torch.randn(5000, 5000, device='cuda')
    alloc = torch.cuda.memory_allocated()
    res = torch.cuda.memory_reserved()
    print(f"After allocating 5000x5000 tensor:")
    print(f"  Allocated: {mb(alloc):.1f} MB")
    print(f"  Reserved:  {mb(res):.1f} MB")
    print(f"  Gap (cache): {mb(res - alloc):.1f} MB")

    del x
    print(f"\nAfter del:")
    print(f"  Allocated: {mb(torch.cuda.memory_allocated()):.1f} MB  (dropped)")
    print(f"  Reserved:  {mb(torch.cuda.memory_reserved()):.1f} MB  (unchanged)")

    torch.cuda.empty_cache()
    print(f"\nAfter empty_cache():")
    print(f"  Reserved:  {mb(torch.cuda.memory_reserved()):.1f} MB  (released)")
else:
    print("GPU not available — memory monitoring tools require CUDA.")
    print("Key APIs: torch.cuda.memory_allocated(), memory_reserved(), memory_summary()")

In [ ]:
if HAS_CUDA:
    print("=== Peak memory tracking ===")
    torch.cuda.reset_peak_memory_stats()

    a = torch.randn(3000, 3000, device='cuda')
    b = torch.randn(3000, 3000, device='cuda')
    c = a @ b  # peak happens here

    peak = torch.cuda.max_memory_allocated()
    print(f"Peak during matmul: {mb(peak):.1f} MB")

    del a, b, c
    print(f"Current after cleanup: {mb(torch.cuda.memory_allocated()):.1f} MB")
    print(f"Peak still recorded: {mb(torch.cuda.max_memory_allocated()):.1f} MB")

    torch.cuda.reset_peak_memory_stats()
    print(f"After reset: peak = {mb(torch.cuda.max_memory_allocated()):.1f} MB")
    torch.cuda.empty_cache()

In [ ]:
if HAS_CUDA:
    print("=== memory_summary() ===")
    x = torch.randn(2000, 2000, device='cuda')
    y = x @ x.T
    print(torch.cuda.memory_summary(abbreviated=True))
    del x, y
    torch.cuda.empty_cache()

In [ ]:
if HAS_CUDA:
    print("=== memory_stats() — programmatic access ===")
    x = torch.randn(1000, 1000, device='cuda')
    stats = torch.cuda.memory_stats()
    for key in ['allocated_bytes.all.current', 'allocated_bytes.all.peak',
                'reserved_bytes.all.current', 'active.all.current',
                'num_alloc_retries', 'num_ooms']:
        val = stats.get(key, 'N/A')
        if isinstance(val, int) and val > 1_000_000:
            print(f"  {key}: {val / 1e6:.1f} MB")
        else:
            print(f"  {key}: {val}")
    del x
    torch.cuda.empty_cache()

## 5. Memory Snapshots

Snapshots capture a complete record of every allocation/deallocation with stack traces.

### Workflow

```python
# 1. Start recording
torch.cuda.memory._record_memory_history(max_entries=100_000)

# 2. Run your code (e.g., one training step)
output = model(batch)
loss = criterion(output, target)
loss.backward()
optimizer.step()

# 3. Save snapshot
torch.cuda.memory._dump_snapshot("snapshot.pickle")

# 4. Stop recording
torch.cuda.memory._record_memory_history(enabled=None)

# 5. Visualize at https://pytorch.org/memory_viz
```

### What you get
- **Segment plot**: memory blocks over time, colored by allocation site
- **Trace plot**: timeline of allocations/deallocations
- **Stack traces**: full Python call stack for each allocation

In [ ]:
if HAS_CUDA:
    print("Recording a memory snapshot...")
    torch.cuda.memory._record_memory_history(max_entries=10_000)

    model = nn.Sequential(
        nn.Linear(512, 1024), nn.ReLU(),
        nn.Linear(1024, 1024), nn.ReLU(),
        nn.Linear(1024, 10),
    ).cuda()

    x = torch.randn(64, 512, device='cuda')
    y = model(x)
    y.sum().backward()

    torch.cuda.memory._dump_snapshot("/tmp/demo_snapshot.pickle")
    torch.cuda.memory._record_memory_history(enabled=None)

    print("Snapshot saved to /tmp/demo_snapshot.pickle")
    print("Upload to https://pytorch.org/memory_viz to visualize")

    del model, x, y
    torch.cuda.empty_cache()
else:
    print("Snapshots require CUDA. See the workflow in the markdown cell above.")

## 6. Common Memory Leaks and Fixes

| Bug | Fix |
|-----|-----|
| `all_losses.append(loss)` | Use `loss.item()` or `loss.detach()` |
| Hidden state not detached | `hidden = hidden.detach()` |
| Closure captures tensor | Capture `.shape` or scalar instead |
| Global cache growing | Use `WeakValueDictionary` or bounded cache |

In [ ]:
# Demo: BAD vs GOOD loss accumulation
model = nn.Linear(64, 10)

# BAD: keeps computation graph alive for every loss
bad_losses = []
for _ in range(5):
    out = model(torch.randn(8, 64))
    loss = out.sum()
    bad_losses.append(loss)  # graph reference held!

print(f"BAD: {len(bad_losses)} losses, all have grad_fn: "
      f"{all(l.grad_fn is not None for l in bad_losses)}")

# GOOD: store only scalars
good_losses = []
for _ in range(5):
    out = model(torch.randn(8, 64))
    loss = out.sum()
    good_losses.append(loss.item())  # scalar, no graph

print(f"GOOD: {len(good_losses)} scalars: {[f'{l:.2f}' for l in good_losses]}")

del bad_losses, good_losses, model

In [ ]:
# Leak detection pattern
def check_for_leak(model, num_steps=10, device='cpu'):
    """Run a few steps and watch memory_allocated."""
    readings = []
    for _ in range(num_steps):
        x = torch.randn(8, 64, device=device)
        out = model(x)
        loss = out.sum()
        loss.backward()
        model.zero_grad(set_to_none=True)
        if HAS_CUDA and device != 'cpu':
            torch.cuda.synchronize()
            readings.append(torch.cuda.memory_allocated())

    if readings:
        growth = readings[-1] - readings[0]
        return growth > readings[0] * 0.1
    return False  # can't check on CPU

model = nn.Linear(64, 10)
if HAS_CUDA:
    model = model.cuda()
    leak = check_for_leak(model, device='cuda')
    print(f"Memory leak detected: {leak}")
else:
    print("Leak detection requires CUDA for memory_allocated() readings")
del model

## 7. Optimization Techniques Walkthrough

### 7a. Gradient Checkpointing

Recompute activations during backward instead of storing them.

- **Saves**: activation memory (O(L) → O(√L) for L layers)
- **Costs**: ~33% more compute

In [ ]:
from torch.utils.checkpoint import checkpoint
import time

dim, num_layers = 256, 8
device = 'cuda' if HAS_CUDA else 'cpu'

model = SimpleLLM(vocab=1000, dim=dim, heads=8, layers=num_layers).to(device)
batch = torch.randint(0, 1000, (8, 64), device=device)

# Without checkpointing
if HAS_CUDA:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
out = model(batch)
out.sum().backward()
t_normal = time.perf_counter() - t0
peak_normal = torch.cuda.max_memory_allocated() / 1e6 if HAS_CUDA else 0

model.zero_grad(set_to_none=True)
if HAS_CUDA:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# With checkpointing
t0 = time.perf_counter()
x = model.embed(batch)
for layer in model.layers:
    x = checkpoint(layer, x, use_reentrant=False)
out = model.head(model.norm(x))
out.sum().backward()
t_ckpt = time.perf_counter() - t0
peak_ckpt = torch.cuda.max_memory_allocated() / 1e6 if HAS_CUDA else 0

print(f"Normal:       time={t_normal*1000:.0f}ms", end="")
if HAS_CUDA: print(f", peak={peak_normal:.0f}MB", end="")
print()
print(f"Checkpointed: time={t_ckpt*1000:.0f}ms", end="")
if HAS_CUDA: print(f", peak={peak_ckpt:.0f}MB  ({(1-peak_ckpt/peak_normal)*100:.0f}% saved)", end="")
print()
print(f"Compute overhead: {(t_ckpt/t_normal - 1)*100:.0f}%")

model.zero_grad(set_to_none=True)
del model

### 7b. Mixed Precision (fp32 vs bf16)

Using `bf16` halves parameter and activation memory.

In [ ]:
# Compare model sizes across dtypes
for label, dtype in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    with torch.device('meta'):
        m = SimpleLLM(vocab=1000, dim=512, heads=8, layers=8)
    elem = torch.tensor([], dtype=dtype).element_size()
    total_mb = sum(p.numel() * elem for p in m.parameters()) / 1e6
    print(f"{label}: {total_mb:.1f} MB")
    del m

# autocast demo
model = nn.Linear(256, 256)
x = torch.randn(8, 256)
print(f"\nWithout autocast: output dtype = {model(x).dtype}")
with torch.autocast(device_type='cpu', dtype=torch.bfloat16):
    print(f"With bf16 autocast: output dtype = {model(x).dtype}")
del model

### 7c. Gradient Accumulation

Large effective batch size without the memory cost.

In [ ]:
model = nn.Linear(64, 10)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

micro_bs = 4
accum_steps = 8
effective_bs = micro_bs * accum_steps
print(f"Micro batch: {micro_bs} | Accum steps: {accum_steps} | Effective batch: {effective_bs}")

optimizer.zero_grad()
total_loss = 0.0
for step in range(accum_steps):
    x = torch.randn(micro_bs, 64)
    out = model(x)
    loss = F.cross_entropy(out, torch.randint(0, 10, (micro_bs,))) / accum_steps
    loss.backward()
    total_loss += loss.item()

optimizer.step()
optimizer.zero_grad()
print(f"Accumulated loss: {total_loss:.4f}")
print(f"Activations used memory proportional to micro_bs={micro_bs}, not effective_bs={effective_bs}")
del model, optimizer

### 7d. In-Place Operations

In [ ]:
# Safe in-place: tensors that don't require grad
x = torch.randn(1000, 1000)
x_id = id(x)
x.relu_()  # in-place, no new allocation
print(f"relu_ in-place: same object = {id(x) == x_id}")

# Unsafe: leaf with requires_grad
leaf = torch.randn(10, requires_grad=True)
try:
    leaf.add_(1)
except RuntimeError as e:
    print(f"leaf.add_(1) error: {e}")

# set_to_none saves memory vs zeroing
model = nn.Linear(100, 100)
out = model(torch.randn(10, 100)).sum()
out.backward()
print(f"\nGrad before zero_grad: norm={model.weight.grad.norm():.4f}")
model.zero_grad(set_to_none=True)
print(f"After zero_grad(set_to_none=True): grad is None = {model.weight.grad is None}")
del model

### 7e. del + gc.collect Pattern

In [ ]:
device = 'cuda' if HAS_CUDA else 'cpu'

big_a = torch.randn(3000, 3000, device=device)
big_b = torch.randn(3000, 3000, device=device)
result = big_a @ big_b

if HAS_CUDA:
    print(f"Before cleanup: {torch.cuda.memory_allocated()/1e6:.0f} MB allocated")

del big_a, big_b  # intermediates no longer needed
gc.collect()
if HAS_CUDA:
    torch.cuda.empty_cache()
    print(f"After del + gc + empty_cache: {torch.cuda.memory_allocated()/1e6:.0f} MB")

# Still have result
print(f"Result shape: {result.shape}")
del result
gc.collect()
if HAS_CUDA:
    torch.cuda.empty_cache()
    print(f"After full cleanup: {torch.cuda.memory_allocated()/1e6:.0f} MB")

## 8. torch.profiler for Memory

The profiler tracks memory alongside compute when `profile_memory=True`.

```python
from torch.profiler import profile, ProfilerActivity

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True,
    with_stack=True,
) as prof:
    output = model(batch)
    loss = criterion(output, target)
    loss.backward()

# Top memory consumers
print(prof.key_averages().table(
    sort_by="self_cuda_memory_usage", row_limit=10
))
```

In [ ]:
from torch.profiler import profile, ProfilerActivity

model = nn.Sequential(
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, 10),
)

x = torch.randn(32, 256)

with profile(
    activities=[ProfilerActivity.CPU],
    profile_memory=True,
    record_shapes=True,
) as prof:
    out = model(x)
    loss = out.sum()
    loss.backward()

print(prof.key_averages().table(sort_by="self_cpu_memory_usage", row_limit=10))

del model

## 9. Memory-Efficient Attention

Standard attention stores the full **N × N** score matrix: O(N²) memory.

Flash Attention computes attention in tiles: **O(N)** memory.

| Sequence Length | Standard Attn | Flash Attn | Savings |
|----------------|--------------|------------|--------|
| 512 | 16 MB | 0.25 MB | 64× |
| 2048 | 256 MB | 1 MB | 256× |
| 4096 | 1 GB | 2 MB | 512× |
| 16384 | 16 GB | 8 MB | 2048× |
| 65536 | 256 GB | 32 MB | — |

In [ ]:
# SDPA automatically uses Flash Attention when available
batch, heads, seq_len, head_dim = 2, 8, 512, 64
q = torch.randn(batch, heads, seq_len, head_dim)
k = torch.randn(batch, heads, seq_len, head_dim)
v = torch.randn(batch, heads, seq_len, head_dim)

out = F.scaled_dot_product_attention(q, k, v)
print(f"SDPA output shape: {out.shape}")

# Manual attention would store the full NxN matrix:
attn_matrix_mb = batch * heads * seq_len * seq_len * 4 / 1e6  # fp32
print(f"Standard attention matrix: {attn_matrix_mb:.1f} MB")
print(f"Flash Attention: avoids materializing this entirely")

## 10. Memory-Efficient Training Checklist

Before training a large model:

1. **Estimate first** — use meta device to compute param + optimizer memory
2. **Pick your precision** — bf16 is free memory savings
3. **Enable Flash Attention** — check SDPA routes to flash kernel
4. **Consider gradient checkpointing** — if activations dominate
5. **Right-size your batch** — use gradient accumulation
6. **Profile before optimizing** — `memory_summary()` or snapshots
7. **Monitor during training** — log `memory_allocated()` per step
8. **Choose optimizer wisely** — Adam = 2× params in state
9. **Test gradually** — increase batch/seq incrementally
10. **Use FSDP for multi-GPU** — shards params, grads, and optimizer

## Exercise: Estimate Memory for a 13B Model

**Task**: Calculate the total GPU memory needed to train a 13B parameter model with:
- bf16 parameters
- AdamW optimizer (fp32 moments + master weights)
- Batch size 4, sequence length 4096
- 40 layers, hidden dim 5120, 40 attention heads
- Flash Attention enabled
- No gradient checkpointing

**Questions**:
1. How much memory for parameters?
2. How much for optimizer state (including master weights)?
3. Will it fit on a single A100-80GB? How many GPUs do you need?
4. How much would gradient checkpointing save on activations?

In [ ]:
# YOUR SOLUTION HERE
# -------------------
num_params = 13_000_000_000

# 1. Parameter memory (bf16 = 2 bytes/param)
param_gb = num_params * 2 / 1e9
print(f"1. Parameters: {param_gb:.1f} GB")

# 2. Gradients
grad_gb = param_gb
print(f"   Gradients: {grad_gb:.1f} GB")

# 3. Optimizer state: Adam m (fp32) + v (fp32) + master weights (fp32)
adam_m = num_params * 4 / 1e9
adam_v = num_params * 4 / 1e9
master = num_params * 4 / 1e9
opt_gb = adam_m + adam_v + master
print(f"2. Optimizer: m={adam_m:.1f}GB + v={adam_v:.1f}GB + master={master:.1f}GB = {opt_gb:.1f} GB")

# 4. Activations
est = estimate_training_memory(
    num_params=13_000_000_000, dtype_bytes=2, optimizer="adam",
    batch_size=4, seq_len=4096, hidden_dim=5120,
    num_layers=40, num_heads=40, flash_attn=True, grad_ckpt=False,
)
print(f"   Activations: {est['activations_gb']:.1f} GB")

total = param_gb + grad_gb + opt_gb + est['activations_gb'] + 0.5
print(f"\n   TOTAL: {total:.1f} GB")
print(f"3. Fits on A100-80GB? {'YES' if total <= 72 else 'NO'}")
print(f"   GPUs needed (A100-80GB): {max(1, math.ceil(total / 72))}")

# With gradient checkpointing
est_ckpt = estimate_training_memory(
    num_params=13_000_000_000, dtype_bytes=2, optimizer="adam",
    batch_size=4, seq_len=4096, hidden_dim=5120,
    num_layers=40, num_heads=40, flash_attn=True, grad_ckpt=True,
)
print(f"\n4. With grad checkpointing: activations {est['activations_gb']:.1f} GB → {est_ckpt['activations_gb']:.1f} GB")
print(f"   Savings: {est['activations_gb'] - est_ckpt['activations_gb']:.1f} GB")

## Key Takeaways

1. **Optimizer state dominates** for large models — Adam uses 2× parameter memory in fp32
2. **Meta device** lets you estimate memory without a GPU — always estimate before training
3. **`memory_allocated()` vs `memory_reserved()`** — the gap is fragmentation/cache
4. **Memory snapshots** are the ultimate debugging tool — full allocation traces
5. **Gradient checkpointing** trades ~33% compute for 60-70% activation memory savings
6. **Mixed precision (bf16)** halves parameter and activation memory for free
7. **Flash Attention** makes long sequences feasible — O(N) vs O(N²)
8. **Profile first, optimize second** — use `memory_summary()` to find the bottleneck
9. **Watch for leaks** — `.item()` instead of appending loss tensors
10. **Combine techniques** — checkpointing + bf16 + accumulation + FSDP for maximum scale

---

<div align="center">

[← Module 25: Triton Kernels](../25_triton_kernels/) | [🏠 Home](../README.md) | Next Module →

</div>